# Log-mel Representation Features (IEMOCAP)

This notebook extracts pooled log-mel spectrogram summaries.
Each utterance becomes one training row for downstream SER models.

In [ ]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [ ]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "representations"
OUT_FILE = "representations_features.csv"

# Audio + feature params
TARGET_SR = 16_000
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 64
FMIN = 50
FMAX = None  # defaults to sr // 2

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


PosixPath('/home/marcello/Speech-Emotion-Recognition/extracted_features/representations/representations_features.csv')

In [ ]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def compute_log_mel(audio: np.ndarray, sr: int) -> np.ndarray:
    # Compute log-mel spectrogram
    fmax = sr // 2 if FMAX is None else FMAX
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=fmax,
        power=2.0,
    )
    return librosa.power_to_db(mel, ref=np.max)


def summarize_matrix(prefix: str, matrix: np.ndarray) -> dict[str, float]:
    # Summary stats across all values in the matrix
    return {
        f"{prefix}_mean": float(matrix.mean()),
        f"{prefix}_std": float(matrix.std()),
        f"{prefix}_min": float(matrix.min()),
        f"{prefix}_max": float(matrix.max()),
        f"{prefix}_median": float(np.median(matrix)),
    }


def extract_representation_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    log_mel = compute_log_mel(audio, sr)
    features: dict[str, float] = {
        "log_mel_frames": float(log_mel.shape[1]),
        "log_mel_bands": float(log_mel.shape[0]),
    }
    features.update(summarize_matrix("log_mel_db", log_mel))

    band_means = log_mel.mean(axis=1)
    band_stds = log_mel.std(axis=1)
    for idx, (mean_val, std_val) in enumerate(zip(band_means, band_stds)):
        features[f"mel_band{idx:02d}_mean_db"] = float(mean_val)
        features[f"mel_band{idx:02d}_std_db"] = float(std_val)
    return features


In [ ]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [ ]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = max(1, CPU_COUNT - 2)
PROGRESS_MIN_INTERVAL = 1.0

print(f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | workers={NUM_WORKERS}")


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_representation_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Saved: /home/marcello/Speech-Emotion-Recognition/extracted_features/representations/representations_features.csv
Workers used: 62 (cpu_count=64)


(7532, 143)